In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df = pd.read_csv("../data/processed/unsw_nb15_processed.csv")

In [ ]:
df['attack_cat'].value_count()

In [ ]:
df['attack_cat'] = df ['attack_cat'].fillna("Normal")

In [ ]:
le = LabelEncoder()
df['attack_cat_encoded'] = le.file_transform(df['attack_cat'])

In [ ]:
attack_mapping = dict(zip(le.classes_, le.transform(df['attack_cat'])

In [ ]:
X = df.drop(columns=['label', 'attack_cat', 'attack_cat_encoded'])
y = df['attack_cat_encoded']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
rf_multi = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_multi.fit(X_train, y_train)

In [ ]:
y_pred = rf_multi.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10,6))
sns.heatmap(
    cm,
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    annot=False,
    cmap="Blues"
)

plt.title("Multi-Class Attack Detection Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
def predict_attack_type(event):
    pred = rf_multi.predict(event.values.reshape(1, -1))[0]
    return le.inverse_transform([pred])[0]

In [ ]:
sample_event = X_test.iloc[10]
print("Predicted Attack Type:", predict_attack_type(sample_event))

In [ ]:
import joblib

joblib.dump(rf_multi, "../models/random_forest_multiclass.pkl")
joblib.dimp(le, "../models/attack_label_encoder.pkl")